# Bank statement import test (KBANK + KTB)

Scans **all Excel files** under:

- `{BASE_DIR}/KBANK`
- `{BASE_DIR}/KTB`

Then imports into Supabase:

- `bank.statement_import_files` (dedupe by `file_hash`)
- `bank.statement_lines` (dedupe by `transaction_fingerprint`)

## Setup

Add to your local `.env` (repo root):

- `SUPABASE_DB_HOST`
- `SUPABASE_DB_PORT`
- `SUPABASE_DB_NAME`
- `SUPABASE_DB_USER`
- `SUPABASE_DB_PASSWORD`

Set `INSERT_TO_DB = True` in the config cell when ready to write.


In [ ]:
# If your environment is missing deps, uncomment and run.
# !pip install -q pandas openpyxl xlrd psycopg2-binary python-dotenv


In [ ]:
import os
import re
import json
import hashlib
from dataclasses import dataclass
from datetime import date
from decimal import Decimal, ROUND_HALF_UP

import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

from pathlib import Path

try:
    from dotenv import load_dotenv

    _cwd = Path.cwd()
    if (_cwd / ".env").exists():
        load_dotenv(_cwd / ".env")
    elif (_cwd.parent / ".env").exists():
        load_dotenv(_cwd.parent / ".env")
except Exception:
    pass


In [ ]:
# --- Configure paths on the worker PC ---
BASE_DIR = r"G:\Shared drives\KCW-Data\kcw_analytics\01_raw\statement"

# Folders to scan (all Excel files inside each folder)
BANK_FOLDERS = ["KBANK", "KTB"]
STATEMENT_EXTENSIONS = {".xlsx", ".xls", ".xlsm"}

# Safety switch: set True when ready to write into Supabase
INSERT_TO_DB = False


In [ ]:
# --- Supabase Postgres connection (.env) ---
# Expected variables (your naming):
#   SUPABASE_DB_HOST
#   SUPABASE_DB_PORT
#   SUPABASE_DB_NAME
#   SUPABASE_DB_USER
#   SUPABASE_DB_PASSWORD
#
# Also accepts legacy PG* names as fallback.

PGHOST = os.getenv("SUPABASE_DB_HOST") or os.getenv("PGHOST", "")
PGPORT = int(os.getenv("SUPABASE_DB_PORT") or os.getenv("PGPORT", "5432"))
PGDATABASE = os.getenv("SUPABASE_DB_NAME") or os.getenv("PGDATABASE", "postgres")
PGUSER = os.getenv("SUPABASE_DB_USER") or os.getenv("PGUSER", "")
PGPASSWORD = os.getenv("SUPABASE_DB_PASSWORD") or os.getenv("PGPASSWORD", "")

if not PGHOST or not PGUSER or not PGPASSWORD:
    print("⚠️  Set SUPABASE_DB_HOST / SUPABASE_DB_USER / SUPABASE_DB_PASSWORD in .env before inserting.")
else:
    print("DB creds found ✅", {"host": PGHOST, "db": PGDATABASE, "user": PGUSER[:20] + "..."})


In [ ]:
def connect_pg():
    return psycopg2.connect(
        host=PGHOST,
        port=PGPORT,
        dbname=PGDATABASE,
        user=PGUSER,
        password=PGPASSWORD,
        sslmode="require",
    )


In [ ]:
def sha256_file_bytes(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def norm_text(x) -> str:
    if x is None or (isinstance(x, float) and pd.isna(x)) or (isinstance(x, pd._libs.missing.NAType)):
        return ""
    s = str(x)
    s = s.replace("\u00A0", " ")
    s = s.strip().upper()
    s = re.sub(r"\s+", " ", s)
    return s

def norm_date(d) -> str:
    if d is None or pd.isna(d):
        return ""
    ts = pd.to_datetime(d, errors="coerce", dayfirst=True)
    if pd.isna(ts):
        return ""
    return ts.date().isoformat()

def norm_money(x) -> str:
    if x is None or pd.isna(x):
        return ""
    # Use Decimal for stable 2dp formatting
    try:
        d = Decimal(str(x).replace(",", "").strip())
    except Exception:
        d = Decimal(str(pd.to_numeric(x, errors="coerce")))
    if d.is_nan():
        return ""
    d = d.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
    return f"{d:.2f}"


In [ ]:
@dataclass
class ParsedLine:
    account_no: str
    bank_name: str | None
    txn_date: date
    value_date: date | None
    description: str | None
    bank_reference: str | None
    amount: Decimal
    direction: str
    debit: Decimal | None
    credit: Decimal | None
    balance_after: Decimal | None
    transaction_fingerprint: str
    source_sheet_name: str | None
    source_row_number: int | None
    raw_json: dict


In [ ]:
def detect_bank_from_path(path: str) -> str | None:
    up = path.upper()
    if "\\KBANK\\" in up or up.endswith("KBANK"):
        return "KBANK"
    if "\\KTB\\" in up or up.endswith("KTB"):
        return "KTB"
    # fallback: folder name
    folder = os.path.basename(os.path.dirname(path)).upper()
    if folder in {"KBANK", "KTB"}:
        return folder
    return None

def _find_header_row(df0: pd.DataFrame) -> int | None:
    """Find a likely header row by scanning for date/desc keywords."""
    # Normalize all cells to string for scanning
    for i in range(min(len(df0), 60)):
        row = df0.iloc[i].astype("string").fillna("")
        joined = "|".join([norm_text(x) for x in row.tolist()])
        hits = 0
        # English keywords
        if "DATE" in joined:
            hits += 1
        if "DESCRIPTION" in joined or "DETAIL" in joined or "PARTICULAR" in joined:
            hits += 1
        if "DEBIT" in joined or "WITHDRAW" in joined:
            hits += 1
        if "CREDIT" in joined or "DEPOSIT" in joined:
            hits += 1
        if re.search(r"\bAMOUNT\b", joined):
            hits += 1
        if "BAL" in joined or "BALANCE" in joined:
            hits += 1
        # Thai keywords (common)
        if "วันที่" in joined:
            hits += 1
        if "รายการ" in joined or "รายละเอียด" in joined:
            hits += 1
        if "เดบิต" in joined or "ถอน" in joined:
            hits += 1
        if "เครดิต" in joined or "ฝาก" in joined:
            hits += 1
        if "คงเหลือ" in joined or "ยอดคงเหลือ" in joined:
            hits += 1

        if hits >= 3:
            return i
    return None

def _norm_cols(cols) -> list[str]:
    out = []
    for c in cols:
        c2 = norm_text(c)
        out.append(c2)
    # de-dup
    seen = {}
    final = []
    for c in out:
        if c not in seen:
            seen[c] = 0
            final.append(c)
        else:
            seen[c] += 1
            final.append(f"{c}_{seen[c]}")
    return final

def _pick_col(cols: list[str], patterns: list[str]) -> str | None:
    for p in patterns:
        rx = re.compile(p)
        for c in cols:
            if rx.search(c):
                return c
    return None


def open_excel_file(path: str) -> pd.ExcelFile:
    """Open Excel; KTB files may use .xls extension but are xlsx (openpyxl)."""
    ext = os.path.splitext(path)[1].lower()
    if ext in {".xlsx", ".xlsm", ".xls"}:
        try:
            return pd.ExcelFile(path, engine="openpyxl")
        except Exception:
            if ext == ".xls":
                return pd.ExcelFile(path, engine="xlrd")
    return pd.ExcelFile(path)


def _extract_account_from_metadata(df0: pd.DataFrame) -> str:
    """Read account number from top rows, e.g. KTB 'Account No.' | '248-0-42113-9'."""
    for i in range(min(len(df0), 20)):
        row = df0.iloc[i].tolist()
        for j, cell in enumerate(row):
            label = norm_text(cell)
            if label in {"ACCOUNT NO.", "ACCOUNT NO", "ACCOUNT NUMBER", "เลขที่บัญชี"}:
                for k in range(j + 1, len(row)):
                    val = row[k]
                    if val is None or (isinstance(val, float) and pd.isna(val)):
                        continue
                    s = str(val).strip()
                    if s:
                        return s
    return ""


def _to_numeric_money(val) -> float | None:
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    s = str(val).replace(",", "").replace("\u00a0", " ").strip()
    if not s or s.upper().startswith("TOTAL"):
        return None
    n = pd.to_numeric(s, errors="coerce")
    if pd.isna(n):
        return None
    return float(n)


def parse_statement_file(path: str, *, bank_name: str | None = None, account_no: str | None = None) -> tuple[dict, list[ParsedLine]]:
    bank_name = bank_name or detect_bank_from_path(path)
    xls = open_excel_file(path)
    meta = {
        "sheet_names": list(xls.sheet_names),
        "parser_version": "auto_v1",
        "bank_name": bank_name,
    }

    lines: list[ParsedLine] = []
    resolved_account = account_no or ""

    for sheet in xls.sheet_names:
        # Load raw grid
        df0 = pd.read_excel(xls, sheet_name=sheet, header=None, dtype=object)
        if df0.empty:
            continue

        header_row = _find_header_row(df0)
        if header_row is None:
            # Not a statement sheet
            continue

        meta_account = _extract_account_from_metadata(df0)
        if meta_account:
            resolved_account = meta_account

        df = pd.read_excel(xls, sheet_name=sheet, header=header_row, dtype=object)
        if df.empty:
            continue

        df.columns = _norm_cols(df.columns)
        cols = list(df.columns)

        # heuristics for common columns
        col_date = _pick_col(cols, [r"^DATE$", r"TXN.*DATE", r"TRAN.*DATE", r"วันที่"]) 
        col_value_date = _pick_col(cols, [r"VALUE.*DATE", r"VAL.*DATE", r"วันที่.*เงิน"]) 
        col_desc = _pick_col(cols, [r"DESC", r"DETAIL", r"PARTICULAR", r"รายการ", r"รายละเอียด"]) 
        col_debit = _pick_col(cols, [r"DEBIT", r"WITHDRAW", r"DR", r"ถอน", r"เดบิต"]) 
        col_credit = _pick_col(cols, [r"CREDIT", r"DEPOSIT", r"CR", r"ฝาก", r"เครดิต"]) 
        col_amount = _pick_col(cols, [r"^AMOUNT$", r"^จำนวนเงิน$"]) 
        col_balance = _pick_col(cols, [r"BAL", r"BALANCE", r"คงเหลือ", r"ยอดคงเหลือ"]) 
        col_ref = _pick_col(cols, [r"REF", r"REFERENCE", r"CHEQUE", r"CHQ", r"เลขที่", r"อ้างอิง", r"^CHEQUE NO"]) 

        if not col_date or (not col_debit and not col_credit and not col_amount):
            continue

        # Keep source row number: approximate row in original excel by offsetting header
        base_row_num = header_row + 2  # 1-based excel row; header is 1 row

        for i, r in df.iterrows():
            raw = {k: (None if pd.isna(v) else v) for k, v in r.to_dict().items()}

            d = pd.to_datetime(r.get(col_date), errors="coerce", dayfirst=True)
            if pd.isna(d):
                continue

            debit = _to_numeric_money(r.get(col_debit)) if col_debit else None
            credit = _to_numeric_money(r.get(col_credit)) if col_credit else None
            signed_amount = _to_numeric_money(r.get(col_amount)) if col_amount else None
            bal = _to_numeric_money(r.get(col_balance)) if col_balance else None

            # Determine direction + amount
            direction = None
            amount = None
            debit_val = None
            credit_val = None

            if credit is not None and credit != 0:
                direction = "in"
                amount = abs(Decimal(str(credit)))
                credit_val = amount
            elif debit is not None and debit != 0:
                direction = "out"
                amount = abs(Decimal(str(debit)))
                debit_val = amount
            elif signed_amount is not None and signed_amount != 0:
                if signed_amount > 0:
                    direction = "in"
                    amount = abs(Decimal(str(signed_amount)))
                    credit_val = amount
                else:
                    direction = "out"
                    amount = abs(Decimal(str(signed_amount)))
                    debit_val = amount
            else:
                continue

            desc = r.get(col_desc) if col_desc else None
            ref = r.get(col_ref) if col_ref else None
            vdate = None
            if col_value_date:
                vd = pd.to_datetime(r.get(col_value_date), errors="coerce", dayfirst=True)
                if not pd.isna(vd):
                    vdate = vd.date()

            # Fingerprint input per your spec
            fp_input = "|".join([
                norm_text(resolved_account),
                norm_date(d),
                norm_money(amount),
                norm_text(direction),
                norm_text(desc),
                norm_text(ref),
                norm_money(bal) if col_balance else "",
            ])
            fp = sha256_text(fp_input)

            lines.append(
                ParsedLine(
                    account_no=resolved_account,
                    bank_name=bank_name,
                    txn_date=d.date(),
                    value_date=vdate,
                    description=None if desc is None or (isinstance(desc, float) and pd.isna(desc)) else str(desc),
                    bank_reference=None if ref is None or (isinstance(ref, float) and pd.isna(ref)) else str(ref),
                    amount=amount,
                    direction=direction,
                    debit=debit_val,
                    credit=credit_val,
                    balance_after=None if bal is None else Decimal(str(bal)),
                    transaction_fingerprint=fp,
                    source_sheet_name=sheet,
                    source_row_number=int(base_row_num + i),
                    raw_json=raw,
                )
            )

    meta["account_no"] = resolved_account
    meta["row_count_detected"] = len(lines)
    return meta, lines


In [ ]:
def list_statement_files(base_dir: str, bank_folders: list[str]) -> list[str]:
    """Return sorted Excel paths under each bank folder."""
    paths: list[str] = []
    for bank in bank_folders:
        folder = os.path.join(base_dir, bank)
        if not os.path.isdir(folder):
            print(f"⚠️  Folder not found: {folder}")
            continue
        for name in sorted(os.listdir(folder)):
            if name.startswith("~$"):  # Excel temp lock files
                continue
            ext = os.path.splitext(name)[1].lower()
            if ext in STATEMENT_EXTENSIONS:
                paths.append(os.path.join(folder, name))
    return paths


def infer_account_from_filename(path: str, bank_name: str | None) -> str:
    """Best-effort account number from filename, e.g. KBANK0393_... -> 0393."""
    base = os.path.splitext(os.path.basename(path))[0].upper()
    if bank_name and base.startswith(bank_name.upper()):
        rest = base[len(bank_name) :]
        m = re.match(r"(\d+)", rest)
        if m:
            return m.group(1)
    m = re.search(r"(\d{3,})", base)
    return m.group(1) if m else ""


In [ ]:
# --- Discover files under KBANK + KTB ---
files = list_statement_files(BASE_DIR, BANK_FOLDERS)
print(f"Found {len(files)} statement file(s) under {BASE_DIR}")
for p in files:
    print(" -", p)

# Quick parse preview for the first file (optional)
if files:
    sample = files[0]
    bank_name = detect_bank_from_path(sample)
    account_no = infer_account_from_filename(sample, bank_name)
    meta, parsed_lines = parse_statement_file(sample, bank_name=bank_name, account_no=account_no)

    print("\nPreview file:", sample)
    print("Bank:", bank_name, "| account_no:", account_no)
    print("Parsed lines:", len(parsed_lines))

    preview = pd.DataFrame([
        {
            "txn_date": x.txn_date,
            "amount": float(x.amount),
            "direction": x.direction,
            "description": (x.description or "")[:60],
            "bank_reference": x.bank_reference,
            "balance_after": None if x.balance_after is None else float(x.balance_after),
            "sheet": x.source_sheet_name,
            "row": x.source_row_number,
        }
        for x in parsed_lines[:20]
    ])
    preview
else:
    print("No files found. Check BASE_DIR and BANK_FOLDERS.")


In [ ]:
def upsert_import_file(conn, *, file_hash: str, original_filename: str, source_path: str, bank_name: str | None, account_no: str | None, raw_metadata: dict):
    """Insert statement_import_files; if duplicate file_hash, just bump last_seen_at and mark duplicate."""
    sql = """
    insert into bank.statement_import_files (
      file_hash, original_filename, source_path, bank_name, account_no,
      status, row_count, inserted_count, duplicate_count, error_count,
      error_message, raw_metadata, first_seen_at, last_seen_at, processed_at
    )
    values (
      %(file_hash)s, %(original_filename)s, %(source_path)s, %(bank_name)s, %(account_no)s,
      'pending', 0, 0, 0, 0,
      null, %(raw_metadata)s::jsonb, now(), now(), null
    )
    on conflict (file_hash) do update
      set last_seen_at = now()
    returning id, (xmax = 0) as inserted;
    """
    with conn.cursor() as cur:
        cur.execute(sql, {
            "file_hash": file_hash,
            "original_filename": original_filename,
            "source_path": source_path,
            "bank_name": bank_name,
            "account_no": account_no,
            "raw_metadata": json.dumps(raw_metadata, ensure_ascii=False),
        })
        file_id, inserted = cur.fetchone()
    conn.commit()
    return file_id, bool(inserted)

def set_file_status(conn, file_id, *, status: str, row_count: int, inserted_count: int, duplicate_count: int, error_count: int, error_message: str | None):
    sql = """
    update bank.statement_import_files
    set
      status = %(status)s,
      row_count = %(row_count)s,
      inserted_count = %(inserted_count)s,
      duplicate_count = %(duplicate_count)s,
      error_count = %(error_count)s,
      error_message = %(error_message)s,
      processed_at = now(),
      last_seen_at = now()
    where id = %(id)s;
    """
    with conn.cursor() as cur:
        cur.execute(sql, {
            "id": file_id,
            "status": status,
            "row_count": row_count,
            "inserted_count": inserted_count,
            "duplicate_count": duplicate_count,
            "error_count": error_count,
            "error_message": error_message,
        })
    conn.commit()


In [ ]:
def process_one_file(conn, path: str, *, insert_to_db: bool) -> dict:
    """Parse one statement file and optionally insert into Supabase."""
    bank_name = detect_bank_from_path(path)
    account_no = infer_account_from_filename(path, bank_name)
    file_hash = sha256_file_bytes(path)

    result = {
        "path": path,
        "bank_name": bank_name,
        "account_no": account_no,
        "file_hash": file_hash,
        "status": "dry_run",
        "row_count": 0,
        "inserted_count": 0,
        "duplicate_count": 0,
        "error_count": 0,
        "error_message": None,
    }

    meta, parsed_lines = parse_statement_file(path, bank_name=bank_name, account_no=account_no)
    result["row_count"] = len(parsed_lines)

    if not insert_to_db:
        return result

    file_id, is_new_file = upsert_import_file(
        conn,
        file_hash=file_hash,
        original_filename=os.path.basename(path),
        source_path=path,
        bank_name=bank_name,
        account_no=account_no or None,
        raw_metadata=meta,
    )

    if not is_new_file:
        result["status"] = "duplicate"
        set_file_status(
            conn,
            file_id,
            status="duplicate",
            row_count=0,
            inserted_count=0,
            duplicate_count=0,
            error_count=0,
            error_message=None,
        )
        return result

    try:
        set_file_status(
            conn,
            file_id,
            status="importing",
            row_count=len(parsed_lines),
            inserted_count=0,
            duplicate_count=0,
            error_count=0,
            error_message=None,
        )

        inserted_count, duplicate_count = insert_statement_lines(conn, file_id=file_id, lines=parsed_lines)
        set_file_status(
            conn,
            file_id,
            status="imported",
            row_count=len(parsed_lines),
            inserted_count=inserted_count,
            duplicate_count=duplicate_count,
            error_count=0,
            error_message=None,
        )

        result["status"] = "imported"
        result["inserted_count"] = inserted_count
        result["duplicate_count"] = duplicate_count
        return result

    except Exception as e:
        set_file_status(
            conn,
            file_id,
            status="failed",
            row_count=len(parsed_lines),
            inserted_count=0,
            duplicate_count=0,
            error_count=1,
            error_message=str(e),
        )
        result["status"] = "failed"
        result["error_count"] = 1
        result["error_message"] = str(e)
        raise


In [ ]:
def insert_statement_lines(conn, *, file_id: str, lines: list[ParsedLine]):
    if not lines:
        return 0, 0

    rows = []
    for x in lines:
        rows.append((
            x.account_no,
            x.bank_name,
            x.txn_date,
            x.value_date,
            x.description,
            x.bank_reference,
            x.amount,
            x.direction,
            x.debit,
            x.credit,
            x.balance_after,
            x.transaction_fingerprint,
            file_id,
            x.source_sheet_name,
            x.source_row_number,
            json.dumps(x.raw_json, ensure_ascii=False),
            "unmatched",
        ))

    insert_sql = """
    insert into bank.statement_lines (
      account_no, bank_name, txn_date, value_date, description, bank_reference,
      amount, direction, debit, credit, balance_after,
      transaction_fingerprint,
      source_file_id, source_sheet_name, source_row_number,
      raw_json, match_status
    )
    values %s
    on conflict (transaction_fingerprint) do nothing
    returning transaction_fingerprint;
    """

    with conn.cursor() as cur:
        execute_values(cur, insert_sql, rows, page_size=500)
        inserted = cur.rowcount  # rowcount is number of returned rows with RETURNING
    conn.commit()

    dupes = len(rows) - inserted
    return inserted, dupes


In [ ]:
# --- Process ALL files in KBANK + KTB ---
results: list[dict] = []

if not files:
    print("Nothing to process.")
elif not INSERT_TO_DB:
    print("Dry-run: parsing only (INSERT_TO_DB=False)")
    for path in files:
        r = process_one_file(None, path, insert_to_db=False)
        results.append(r)
        print(f"[dry_run] {os.path.basename(path)} -> rows={r['row_count']}")
else:
    conn = connect_pg()
    try:
        for path in files:
            print(f"Processing: {path}")
            try:
                r = process_one_file(conn, path, insert_to_db=True)
                results.append(r)
                print(
                    f"  -> {r['status']} | rows={r['row_count']} "
                    f"inserted={r['inserted_count']} dup_txn={r['duplicate_count']}"
                )
            except Exception as e:
                results.append({
                    "path": path,
                    "status": "failed",
                    "error_message": str(e),
                })
                print(f"  -> failed: {e}")
    finally:
        conn.close()

summary = pd.DataFrame(results)
print("\n=== Summary ===")
summary
